# 18.13 圖片／音訊能力能如何蒸餾？

# 第 18 章：蒸餾，讓小模型向教師學習

前置：CE、模型輸出與學生訓練。教師像經驗較多的老師：可以只給答案，也可展示對各候選的分配。老師可能說錯；學生也須真的選較小架構。此處只驗loss與梯度，正式蒸餾需讀者提供已訓練教師。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一個圖片佔位格，展開成三個向量**

展開 labels 同步加 ignored，再做下一字 shift。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/modal_expand.svg")))

**先想一想：**老師16個視覺token、學生4個，直接KL整條序列行嗎？

教師與學生看同一圖片／聲音／問題，比較相同回答位置的分佈。先保持模態展開長度契約；不同視覺token數時不能逐格盲對齊。

**把直覺寫成數學：**文字答案positions必須映射到同一語義token；內部feature distillation另設目標。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
teacher_visual, student_visual, answer_tokens = 16, 4, 3
teacher_answer_positions = list(range(teacher_visual, teacher_visual + answer_tokens))
student_answer_positions = list(range(student_visual, student_visual + answer_tokens))
print(teacher_answer_positions, student_answer_positions)
print("先按對應答案位置gather，再比較同一詞表分佈")

**如何讀結果：**這個索引表不含role等額外位置，真實映射從expand labels的有效位置取得。

**只改一件事：**只改音訊frame數，檢查同樣需要答案位置映射。
